# N-Gram Language Models
Author: Alexander Dao <br>
**In the interest of time, parts of this project are adapted from Class 8b's lab design.**

In [5]:
from collections import Counter, defaultdict
import random
import re
import requests
import spacy

nlp = spacy.blank("en")
nlp.max_length = 5_000_000

After we import all the dependencies, we set the article we want to use. We'll be using Wikipedia for the sake of time and prototyping.

In [6]:
url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query", # action to be done
    "format": "json", # desired return format
    "titles": "Google", # article to query. ONLY modify the value for "titles"
    "prop": "extracts", # specify for summary of article
    "explaintext": True # this makes sure that we're getting text not html slop
}
headers = {"User-Agent": "LING-144 classroom notebook (educational use)"}
response = requests.get(url, params = params, headers = headers, timeout = 30)
response.raise_for_status()

response_text = response.json() # python dict format
pages = response_text["query"]["pages"]
page_ID = list(pages.keys())[0] # fetch page ID
cleaned_text = pages[page_ID]["extract"]

print(cleaned_text[0:150]) # this should return cleaned text from wikipedia

Google LLC ( , GOO-gəl) is an American multinational technology corporation focused on information technology, online advertising, search engine techn


We should now have cleaned text from Wikipedia that we can then process using SpaCy. We'll tokenize and remove case from words so we can get an accurate idea of what's present.

In [7]:
doc = nlp(cleaned_text)
tokens = [tok.text.lower() for tok in doc if not tok.is_space and tok.is_alpha]
print(tokens[:500])

['google', 'llc', 'goo', 'gəl', 'is', 'an', 'american', 'multinational', 'technology', 'corporation', 'focused', 'on', 'information', 'technology', 'online', 'advertising', 'search', 'engine', 'technology', 'email', 'cloud', 'computing', 'software', 'quantum', 'computing', 'e', 'commerce', 'consumer', 'electronics', 'and', 'artificial', 'intelligence', 'ai', 'it', 'has', 'been', 'referred', 'to', 'as', 'the', 'most', 'powerful', 'company', 'in', 'the', 'world', 'by', 'the', 'bbc', 'and', 'is', 'one', 'of', 'the', 'world', 'most', 'valuable', 'brands', 'google', 'parent', 'company', 'alphabet', 'has', 'been', 'described', 'as', 'a', 'big', 'tech', 'company', 'google', 'was', 'founded', 'in', 'by', 'american', 'computer', 'scientists', 'larry', 'page', 'and', 'sergey', 'brin', 'together', 'they', 'own', 'about', 'of', 'its', 'publicly', 'listed', 'shares', 'and', 'control', 'of', 'its', 'stockholder', 'voting', 'power', 'through', 'super', 'voting', 'stock', 'the', 'company', 'went', 'pu

Once we've got all the words tokenized and any remaining punctuation/whitespace cleaned up, we can build the LM.

In [8]:
try:
  n = int(input("Enter a number up to 4\n"))
except ValueError:
  print("Invalid input")
else:
  if n < 0:
    print("Invalid input")
  if n > 4:
    print("N-grams above 4 not supported currently.")

# KEEP THIS IN MIND: THIS HASN'T BEEN IMPLEMENTED YET.
# REASON BEING THAT THE ORIGINAL CODE FROM LAB 8B
# IS ACTUALLY HARDCODED FOR BI TRI AND QUADRIGRAMS

Enter a number up to 4
4


In [9]:
def build_ngram_model(tokens, n):
    model = defaultdict(Counter)

    for i in range(0, len(tokens)-1):
      history = tuple(tokens[i-(n-1):i])
      next = tokens[i]
      model[history][next] += 1
    return model

bigram_model = build_ngram_model(tokens, 2)
trigram_model = build_ngram_model(tokens, 3)
quadrigram_model = build_ngram_model(tokens, 4)

print(f"Bigram contexts: {len(bigram_model):,}")
print(f"Trigram contexts: {len(trigram_model):,}")
print(f"Quadrigram contexts: {len(quadrigram_model):,}")

Bigram contexts: 2,562
Trigram contexts: 7,649
Quadrigram contexts: 9,485


Once the model is built, we can then generate text. The following script generates the next token based on the frequency count associated with it.

In [10]:
def weighted_choice(counter):
    choices = list(counter.keys())
    weights = list(counter.values())
    return random.choices(choices, weights=weights, k=1)[0]

def generate_tokens(model, n, number_of_tokens, start=()):
    start = tuple(word.lower() for word in start)

    if len(start) >= n - 1:
        generated = list(start)
    else:
        possible_contexts = [context for context in model if context[:len(start)] == start]
        if not possible_contexts:
            raise ValueError("Those starting words do not occur as a context in this book.")
        generated = list(random.choice(possible_contexts))

    while len(generated) < number_of_tokens:
        context = tuple(generated[-(n - 1):])
        choices = model.get(context)

        # A context at the very end of the book may have no continuation.
        # In that rare case, begin again from another context of the same model.
        if not choices:
            restart = random.choice(list(model.keys()))
            generated.extend(restart)
        else:
            generated.append(weighted_choice(choices))

    return generated[:number_of_tokens]

def untokenize(tokens):
    # Join tokens, then remove spaces before common punctuation.
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:%\)\]’])", r"\1", text)
    text = re.sub(r"([\(\[‘])\s+", r"\1", text)
    text = re.sub(r"\s+(['’])\s+", r"\1", text)
    return text


In [11]:
model_to_use = "quadrigram" #@param ["bigram", "trigram", "quadrigram"]
number_of_tokens = 100 #@param {type:"integer"}
starting_words = "The" #@param {type:"string"}
random_seed = 6 #@param {type:"integer"}

random.seed(random_seed)
start = tuple(starting_words.lower().split())

if model_to_use == "bigram":
    generated = generate_tokens(bigram_model, 2, number_of_tokens, start=start)
elif model_to_use == "trigram":
    generated = generate_tokens(trigram_model, 3, number_of_tokens, start=start)
else:
    generated = generate_tokens(quadrigram_model, 4, number_of_tokens, start=start)

print(untokenize(generated))

the wage gap was around and that google locked women into lower career tracks leading to smaller salaries and bonuses in june google announced google cardboard a simple cardboard viewer that lets the user place their smartphone in the front hinge to view vr media other hardware products include nest a series of voice assistant smart speakers that can answer voice queries play music find information from apps calendar weather etc and control third party smart home appliances users can tell it to turn on the lights for example the google nest line includes the original google home later succeeded
